### Generate input YAML files Boltz2 

Basic goal is to simply utilize Botlz2 for protein-ligand prediction. Since previous structures have been generated via AF3, the goal is to recycle the structure as well as the msa to skip structure prediction for botlz2. 

https://github.com/jwohlwend/boltz?tab=readme-ov-file
https://github.com/jwohlwend/boltz/blob/main/docs/prediction.md

In [1]:
import numpy as np 
import pandas as pd 
import os 
import json 

In [25]:
import json
import os

def extract_unpaired_msa_to_a3m(json_path, output_dir, chain='A'):
    # Load AF3 data
    with open(json_path, 'r') as f:
        data = json.load(f)

    sequences = data.get("sequences", [])
    saved = 0

    for seq in sequences:
        protein = seq.get("protein", {})
        unpaired_msa = protein.get("unpairedMsa", None)
        protein_id = protein.get("id", None)

        if unpaired_msa and (protein_id == chain):
            # Handle case where ID is list or string
            if isinstance(protein_id, list):
                protein_id = protein_id[0]

            # Save to .a3m
            a3m_id = data['name'].split('_')[0].lower()
            output_path = os.path.join(output_dir, f"{a3m_id}.a3m")
            with open(output_path, "w") as f:
                f.write(unpaired_msa.strip() + "\n")
            print(f"Saved {output_path}")
            saved += 1

    if saved == 0:
        print("No unpairedMsa found in any protein entries.")


In [29]:
"""
Generate a3m unpaired msa alignments for OR from AF3 calculated .json files
"""

from tqdm import tqdm 

af3_dir = '/mnt/data2/Justice/AF3_files/AF3_out/from_Dan/consor4/output'
af3_or_dir = os.listdir(af3_dir)
af3_or_dir = [_file for _file in af3_or_dir if not _file.startswith('.')]

n = 0
for _dir in tqdm(af3_or_dir): 
    # only write a3m file if it does not exist
    if ~np.any([True for _file in os.listdir(os.path.join(af3_dir, _dir)) if _file.endswith('.a3m')]): 
        extract_unpaired_msa_to_a3m(f'{os.path.join(af3_dir, _dir, _dir)}_data.json', 
                                    os.path.join(af3_dir, _dir))
        n += 1
    

print(f'Total {len(af3_or_dir)}, generated a3m for {n} / {len(af3_or_dir)}')
    

 50%|█████     | 1/2 [00:00<00:00,  7.88it/s]

Saved /mnt/data2/Justice/AF3_files/AF3_out/from_Dan/consor4/output/consor4_2-methylthiazoline_oronly_pred/consor4.a3m


100%|██████████| 2/2 [00:00<00:00,  3.98it/s]

Saved /mnt/data2/Justice/AF3_files/AF3_out/from_Dan/consor4/output/consor4_2-methylthiazoline_pred/consor4.a3m
Total 2, generated a3m for 2 / 2
